In [ ]:
import os
import sys

%load_ext autoreload
%autoreload 2
import logging
# Configure logging
logging.basicConfig(level=logging.INFO)

# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))
sys.path.append(src_path)

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
import pickle
from server_config import datapath, preprocessed_path_freezed, redcap_path, preprocessed_path
from functions.preprocessing.ema_mappings import clean_heart_rate_data


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt



In [ ]:
from functions.preprocessing import gps_features
from functions.preprocessing.ema_mappings import run_ema_mappings
from functions.preprocessing.missing_data import summarize_missing_data

In [ ]:
import warnings
# Suppress only SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [ ]:
def plot_cdf(x, bins=1000, cumulative=True, density=True, ylabel="Quantile", **kwargs):
    plt.hist(x, bins=bins, cumulative=cumulative, density=density, histtype='step', **kwargs)
    plt.ylabel(ylabel)

In [ ]:
# backup_path = preprocessed_path_freezed + "/backup_data_passive_actual.feather"
backup_path = "/sc-projects/sc-proj-cc15-preact/SP6/raw/backup_passive_recent.feather" # new file 
df_backup = pd.read_feather(backup_path)
print(df_backup.shape)
df_backup.head()

In [ ]:
# # take only the subset of customers
# # comment it out when running on full data
# keep_customers_for_fast_iteration = df_backup.customer.unique()[-25:]
# df_backup = df_backup[df_backup.customer.isin(keep_customers_for_fast_iteration)]
# print(df_backup.shape)
# df_backup.head()

In [ ]:
df_backup

In [ ]:
# TODO move it to 01_data_preprocess, as it takes some time to run, and makes more sense there

df_backup["local_start_time"] = (
    df_backup["startTimestamp"]
    + pd.to_timedelta(df_backup["inferred_tzoffset"], unit="min")
).dt.tz_localize(None)

df_backup["local_end_time"] = (
    df_backup["endTimestamp"]
    + pd.to_timedelta(df_backup["inferred_tzoffset"], unit="min")
).dt.tz_localize(None)

In [ ]:
df_backup

In [ ]:
# print(df_backup[df_backup["timezoneOffset"].isna()]["type"].value_counts())
# df_backup["type"].value_counts().sort_index()["AtrialFibrillationDetection"]
# all the missing timezoneOffsets are AtrialFibrillationDetection,
# and all the AtrialFibrillationDetection miss timezoneOffsets

In [ ]:
# (df_backup["timezoneOffset"] / 60).value_counts().sort_index()


timezoneOffset isn't with us anymore, so the following don't work

In [ ]:
# Plot histogram of timezone offsets (in hours) with bins centered on full hours
timezone_hours = df_backup["timezoneOffset"] / 60
timezone_hours_clean = timezone_hours.dropna()

plt.figure(figsize=(12, 6))
bins = np.arange(-12.875, 13.875, 0.25) 
plt.hist(timezone_hours_clean, bins=bins, edgecolor='black', alpha=0.7)
plt.xlabel('Timezone Offset (hours)')
plt.ylabel('Count')
plt.yscale("log")
plt.title('Distribution of Timezone Offsets')
plt.xticks(range(-12, 14))
plt.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

In [ ]:
df_backup["t_day"] = df_backup["startTimestamp"].dt.floor("D")
df_backup["t_day"]

In [ ]:
df_backup["t_day"].dtype

In [ ]:
df_customer_day_nunique_tz_offsets = df_backup.groupby(["customer", "t_day"])["timezoneOffset"].nunique()

In [ ]:
(df_customer_day_nunique_tz_offsets > 1).sum() / df_customer_day_nunique_tz_offsets.size

In [ ]:
df_backup["t_day"]

In [ ]:
pd.to_datetime("2024-01-16")

In [ ]:
df_backup[
    (df_backup["customer"] == "bruD")
    & (df_backup["t_day"] == pd.to_datetime("2024-01-16", utc=True))
].sort_values("startTimestamp")[["startTimestamp", "timezoneOffset", "type"]]

In [ ]:
df_backup[
    (df_backup["customer"] == "bruD")
    & (df_backup["t_day"] >= pd.to_datetime("2024-01-13", utc=True))
    & (df_backup["t_day"] <= pd.to_datetime("2024-01-18", utc=True))
].groupby(["type", "t_day"])["timezoneOffset"].unique().unstack(
    level="t_day", fill_value=[]
)

In [ ]:
df_backup["local_startTimestamp"] = (
    df_backup["startTimestamp"]
    + pd.to_timedelta(df_backup["timezoneOffset"], unit="min")
).dt.tz_localize(None)

df_backup["local_endTimestamp"] = (
    df_backup["endTimestamp"]
    + pd.to_timedelta(df_backup["timezoneOffset"], unit="min")
).dt.tz_localize(None)

In [ ]:
df_backup

In [ ]:
customers_with_100k_additional_measured_HR_seconds = ['N3CY', 'OmAV', 'DRtE', 'EtmK', 'G2Gk', 'lAHE', '4MLe', 'ZAjp', 'tilE', '0xWn', 'JIhV', 'WF9K', 'AfEo', 'g6p5', 'iIYT']
if False:
    df_backup = df_backup[~df_backup["customer"].isin(customers_with_100k_additional_measured_HR_seconds)]
df_backup

In [ ]:
df_backup["customer"].nunique() 

In [ ]:
# df_backup.groupby(["type"])["customer"].nunique().sort_values(ascending=False)
df_backup.groupby(["type"])["customer"].nunique().sort_index()

In [ ]:
df_backup["duration"] = (df_backup["endTimestamp"] - df_backup["startTimestamp"]).dt.total_seconds()
# 1 sec min and 24h max, mostly 60sec

df_backup["duration"].describe([0.01, 0.05, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99])

In [ ]:
df_backup["customer"].nunique()

In [ ]:
# Convert the result to a DataFrame and print as markdown table
result = df_backup.groupby("type")["customer"].nunique().sort_values(ascending=False)
result_df = result.reset_index()
result_df.columns = ['Data Type', 'Unique Customers']
# print(result_df.to_markdown(index=False))
result

sums like below don't work, as the events overlap, so we overcount

In [ ]:
df_backup[df_backup["type"] == "ActiveBurnedCalories"].groupby(
    ["customer", "startTimestamp_day"]
)["doubleValue"].sum().sort_values(ascending=False)

In [ ]:
target_date = pd.to_datetime("2024-05-05")
target_date = pd.to_datetime("2023-12-11")
# df_backup[(df_backup["customer"] == "1BAf") & (df_backup["startTimestamp_day"] == target_date)]
df_backup[(df_backup["customer"] == "1BAf") & (df_backup["startTimestamp_day"] == target_date) & (df_backup["type"] == "ActiveBurnedCalories")]
# df_backup[(df_backup["customer"] == "lAHE") & (df_backup["type"] == "ActiveBurnedCalories")]
# df_backup[(df_backup["customer"] == "lAHE") & (df_backup["startTimestamp_day"] == target_date)]
# df_backup[(df_backup["customer"] == "lAHE") & (df_backup["type"] == "ActiveBurnedCalories")]
# first record says that 17k calories were burned on that day something is off with this 1BAf customer

# HeartRate

to get rid of overlapping events, we need to "expand" the dataset, so that one row last only one second (the smalles time duration)

In [ ]:
# df = df_backup[(df_backup["type"] == "HeartRate") & (df_backup["customer"].isin(["4MLe","kVhY"]))]
df = df_backup[(df_backup["type"] == "HeartRate")]
df = df[["customer", "startTimestamp", "endTimestamp", "longValue"]]
df = df.rename(columns={"longValue": "HeartRate"})


In [ ]:
df["duration"] = (df["endTimestamp"] - df["startTimestamp"]).dt.total_seconds().fillna(0).astype(int)
df["duration"].describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
df_expanded = df.loc[df.index.repeat(df['duration'])].copy()
df_expanded

In [ ]:
df_expanded['time_offset'] = df_expanded.groupby(level=0).cumcount()
df_expanded['timestamp'] = df_expanded['startTimestamp'] + pd.to_timedelta(df_expanded['time_offset'], unit='s')

as we can see, there are over 1M overlapping entries

In [ ]:
# df_expanded_groupby = df_expanded.groupby(['customer', 'timestamp'])
df_expanded_groupby = df_expanded.groupby(['timestamp', 'customer'])

df_expanded_groupby.size().value_counts().sort_index()

In [ ]:
df_size = df_expanded_groupby.size().reset_index(name='n_repeat')
# drop all the rows where count is 1
df_size = df_size[df_size['n_repeat'] > 1]
df_size.sort_values(by='n_repeat', ascending=False)


In [ ]:
df_size["n_repeat"].describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99])

In [ ]:
df_size["n_additional"] = df_size["n_repeat"] - 1
df_size.groupby("customer")["n_additional"].sum().sort_values(ascending=False)
customer_additional = df_size.groupby("customer")["n_additional"].sum().sort_values(ascending=False)
# Get all customers from df_backup and ensure they're included with 0 if not in customer_additional
all_customers = df_backup["customer"].unique()
customer_additional = customer_additional.reindex(all_customers, fill_value=0).sort_values(ascending=False)

print(customer_additional.describe([]).round(2))
customer_additional.quantile([0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99, 1.0], interpolation="linear").round(2)

by setting threshold of n_additional = 100_000 sec, we keep 97% of patients 

In [ ]:
plt.figure(figsize=(12, 8))
plt.bar(range(len(customer_additional)), customer_additional.values)

# Create percentage-based x-axis labels (reversed from 100% to 0%)
n_customers = len(customer_additional)
percentage_ticks = np.arange(0, n_customers, max(1, n_customers // 20))  # Show ~20 ticks
percentage_labels = [f"{100 - (i/n_customers)*100:.0f}%" for i in percentage_ticks]

plt.xticks(percentage_ticks, percentage_labels)
plt.xlabel('Patients Percentile (100% = highest duplicates, 0% = lowest duplicates)')
plt.ylabel('Number of Additional Duplicate Heart Rate Measurements (seconds)')
plt.title('Additional Duplicate Heart Rate Measurements by Customer')
plt.grid(True, alpha=0.3)
# plt.yscale('log')  # Use logarithmic scale for better visibility

# Add horizontal line at 100,000 additional measurements
plt.axhline(y=100000, color='red', linestyle='--', linewidth=2, label='100,000 threshold')

# Find the position where customer_additional crosses 100,000
threshold_index = (customer_additional > 100000).sum()
threshold_percentage = 100 - (threshold_index / n_customers) * 100

# Add vertical line at the threshold position
plt.axvline(x=threshold_index, color='red', linestyle='--', linewidth=2, alpha=0.7)

# Add text annotation
plt.text(threshold_index + 20, 100000 * 1.5, f'{threshold_percentage:.1f}% of patients\nhave <100k duplicates each', 
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.legend()
plt.tight_layout()

print(f"Threshold analysis:")
print(f"Number of customers with >100,000 additional duplicates: {threshold_index}")
print(f"Percentage of customers with >100,000 additional duplicates: {threshold_percentage:.1f}%")
print(f"Percentage of customers with ≤100,000 additional duplicates: {100 - threshold_percentage:.1f}%")

# Also show top 20 customers with most duplicates
print("\nTop 20 customers with most additional duplicates:")
customer_additional_top20 = customer_additional.head(20)
print(customer_additional_top20)

plt.xlim(-1, len(customer_additional)/2)  # Set x-axis limits to cover all customers
plt.show()


# Print customer names with >100,000 additional duplicates for easy copying
customers_over_threshold = customer_additional[customer_additional > 100000].index.tolist()
print(f"Customers with >100,000 additional duplicates ({len(customers_over_threshold)} total):")
print(customers_over_threshold)

In [ ]:
# Remove duplicates by taking the average where there are multiple entries per timestamp

df_avg = df_expanded_groupby.agg({
    'HeartRate': 'mean',
}).reset_index()

df_min = df_expanded_groupby.agg({
    'HeartRate': 'min',
}).reset_index()

df_max = df_expanded_groupby.agg({
    'HeartRate': 'max',
}).reset_index()

df_avg

In [ ]:
diff = df_max["HeartRate"] - df_min["HeartRate"]
print(f"proportion of repeated data that have different values: {(diff>0).sum() / (len(df_expanded) - len(df_avg)):.2%}")
print("out of this, the following are the statistics of the differences:")
diff[diff > 0].describe([0.25,0.5,0.75,0.9, 0.95, 0.99])
# TODO resolve what to do with this next, about ~80 of repeated 

## HeartRate aggregation

In [ ]:
# df = df_avg[df_avg["customer"].isin(["1BAf", "lAHE", "4MLe", "kVhY"])]
df_hr = df_avg

In [ ]:
df_hr["HR_zone_resting"] = df_hr["HeartRate"] < 60
df_hr["HR_zone_moderate"] = (60 <= df_hr["HeartRate"]) & (df_hr["HeartRate"] < 100)
df_hr["HR_zone_vigorous"] = (100 <= df_hr["HeartRate"])

In [ ]:
# Create hourly heart rate averages
df_hr['t_hour'] = df_hr['timestamp'].dt.floor('h')
df_hr_hourly = df_hr.groupby(['customer', 't_hour']).agg(
    HR_count=('HeartRate', 'count'),
    HR_mean=('HeartRate', 'mean'),
    HR_std=('HeartRate', 'std'),
    HR_min=('HeartRate', 'min'),
    HR_max=('HeartRate', 'max'),
    HR_skew=('HeartRate', "skew"),
    HR_kurtosis=('HeartRate', lambda x: x.kurtosis()),
    HR_zone_resting=('HR_zone_resting', 'sum'),
    HR_zone_moderate=('HR_zone_moderate', 'sum'),
    HR_zone_vigorous=('HR_zone_vigorous', 'sum'),
).reset_index()

df_hr_hourly

In [ ]:
# Create daily heart rate averages
df_hr['t_day'] = df_hr['timestamp'].dt.floor('d')
df_hr_daily = df_hr.groupby(['customer', 't_day']).agg(
    HR_count=('HeartRate', 'count'),
    HR_mean=('HeartRate', 'mean'),
    HR_std=('HeartRate', 'std'),
    HR_min=('HeartRate', 'min'),
    HR_max=('HeartRate', 'max'),
    HR_skew=('HeartRate', "skew"),
    HR_kurtosis=('HeartRate', lambda x: x.kurtosis()),
    HR_zone_resting=('HR_zone_resting', 'sum'),
    HR_zone_moderate=('HR_zone_moderate', 'sum'),
    HR_zone_vigorous=('HR_zone_vigorous', 'sum'),
).reset_index()

df_hr_daily

In [ ]:
# this stopped working TypeError: Invalid comparison between dtype=datetime64[ns, UTC] and Timestamp
# df_plot = df_avg[(df_avg["customer"] == "05kz") & (df_avg["timestamp"] >= pd.to_datetime("2023-10-19")) 
#                  & (df_avg["timestamp"] < pd.to_datetime("2023-10-22"))]
# df_plot

# plt.figure(figsize=(12, 6))
# plt.scatter(df_plot['timestamp'], df_plot['HeartRate'], linewidth=0.5) # scatter vs line plot
# plt.title(f'Heart Rate Data for Customer {df_plot["customer"].iloc[0]} on {df_plot["timestamp"].dt.date.iloc[0]} onwards')
# plt.xlabel('Time')
# plt.ylabel('Heart Rate (BPM)')
# plt.xticks(rotation=45)
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

# Steps

In [ ]:
# Similar to HeartRate analysis, expand the Steps dataset
df_steps_original = df_backup[(df_backup["type"] == "Steps")]
df_steps_original = df_steps_original[
    [
        "customer",
        "for_id",
        "timezoneOffset",
        "startTimestamp",
        "endTimestamp",
        "local_startTimestamp",
        "local_endTimestamp",
        "doubleValue",
    ]
]
df_steps_original = df_steps_original.rename(columns={"doubleValue": "Steps"})
df_steps_original

In [ ]:
print(f"{(df_steps_original["startTimestamp"].dt.second != 0).sum() / len(df_steps_original):.2%}")

In [ ]:
bins = np.arange(0, 61, 1) - 0.5  # 0 to 59 seconds
df_steps_original["startTimestamp"].dt.second.describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99])
plt.figure(figsize=(10, 6))
plt.hist(df_steps_original["startTimestamp"].dt.second, bins=bins, alpha=0.5, edgecolor='black')
plt.hist(df_steps_original["endTimestamp"].dt.second, bins=bins, alpha=0.5, edgecolor='black')
plt.title('Distribution of Start Timestamp Seconds for Steps Data')
plt.xlabel('Second (0-59)')
plt.ylabel('Frequency')
plt.yscale('log')  # Use logarithmic scale for better visibility
plt.grid(True, alpha=0.3)
plt.xticks(np.arange(0, 60, 5))
plt.tight_layout()
plt.show()

such a samll number of samples got different offset than 0 (less than 2%), and 99% of samples durations are withing (59,61) range -- more than 90% is exactly 60s

therefore, just round the seconds 

In [ ]:
df_steps_original["duration"] = (df_steps_original["endTimestamp"] - df_steps_original["startTimestamp"]).dt.total_seconds().fillna(0).astype(int)
df_steps_original["duration"].describe([0.003, 0.004, 0.005, 0.0075, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.993, 0.994, 0.995, 0.9975, 0.999])

In [ ]:
mask_under_59 = df_steps_original["duration"] < 59
mask_over_61 = df_steps_original["duration"] > 61
print(f"Percentage of samples with duration not in [59, 61]: {((mask_under_59 | mask_over_61).sum() / len(df_steps_original)) * 100:.2f}%")
df_steps_original["SPM"] = df_steps_original["Steps"] / (df_steps_original["duration"] / 60) # StepsPerMinute (SPM)
df_steps_original["SPM"].describe([0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99, 0.995])

In [ ]:
df_steps_original["SPM"][mask_under_59].describe([0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99, 0.995])

In [ ]:
df_steps_original["SPM"][mask_over_61].describe([0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99, 0.995])

In [ ]:
df_steps_original["startTimestamp_minute"] = df_steps_original["startTimestamp"].dt.round('min')
# (df_steps["duration"] / 60).round(0).astype
df_steps_original["duration_minutes"] = np.maximum(1, (df_steps_original["duration"] / 60).round(0).astype(int))
df_steps_original["duration_minutes"].min()

In [ ]:
df_steps_expanded = df_steps_original.loc[df_steps_original.index.repeat(df_steps_original['duration_minutes'])].copy()
df_steps_expanded

In [ ]:
df_steps_expanded['time_offset'] = df_steps_expanded.groupby(level=0).cumcount()
df_steps_expanded['timestamp'] = df_steps_expanded['startTimestamp'] + pd.to_timedelta(df_steps_expanded['time_offset'], unit='min')
df_steps_expanded

In [ ]:
# Check for overlapping entries in Steps data
df_steps_expanded_groupby = df_steps_expanded.groupby(['customer','timestamp'])

df_steps_expanded_groupby.size().value_counts().sort_index()

In [ ]:
df_steps_size = df_steps_expanded_groupby.size().reset_index(name='n_repeat')
# drop all the rows where count is 1
df_steps_size = df_steps_size[df_steps_size['n_repeat'] > 1]
df_steps_size.sort_values(by='n_repeat', ascending=False)

In [ ]:
df_steps_size["n_repeat"].describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99])

In [ ]:
df_steps_size["n_additional"] = df_steps_size["n_repeat"] - 1
df_steps_size.groupby("customer")["n_additional"].sum().sort_values(ascending=False)
customer_steps_additional = df_steps_size.groupby("customer")["n_additional"].sum().sort_values(ascending=False)
# Get all customers from df_backup and ensure they're included with 0 if not in customer_steps_additional
all_customers = df_backup["customer"].unique()
customer_steps_additional = customer_steps_additional.reindex(all_customers, fill_value=0).sort_values(ascending=False)

print(customer_steps_additional.describe([]).round(2))
customer_steps_additional.quantile([0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99, 1.0], interpolation="linear").round(2)

In [ ]:
plt.figure(figsize=(12, 8))
plt.bar(range(len(customer_steps_additional)), customer_steps_additional.values)

# Create percentage-based x-axis labels (reversed from 100% to 0%)
n_customers = len(customer_steps_additional)
percentage_ticks = np.arange(0, n_customers, max(1, n_customers // 20))  # Show ~20 ticks
percentage_labels = [f"{100 - (i/n_customers)*100:.0f}%" for i in percentage_ticks]

plt.xticks(percentage_ticks, percentage_labels)
plt.xlabel('Patients Percentile (100% = highest duplicates, 0% = lowest duplicates)')
plt.ylabel('Number of Additional Duplicate Steps Measurements (seconds)')
plt.title('Additional Duplicate Steps Measurements by Customer')
plt.grid(True, alpha=0.3)

# Add horizontal line at 100,000 additional measurements
# plt.axhline(y=100000, color='red', linestyle='--', linewidth=2, label='100,000 threshold')

# Find the position where customer_steps_additional crosses 100,000
threshold_index = (customer_steps_additional > 100000).sum()
threshold_percentage = 100 - (threshold_index / n_customers) * 100

# Add vertical line at the threshold position
# plt.axvline(x=threshold_index, color='red', linestyle='--', linewidth=2, alpha=0.7)

# Add text annotation
# plt.text(threshold_index + 20, 100000 * 1.5, f'{threshold_percentage:.1f}% of patients\nhave <100k duplicates each', 
        #  bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.legend()
plt.tight_layout()

print(f"Steps Threshold analysis:")
print(f"Number of customers with >100,000 additional duplicates: {threshold_index}")
print(f"Percentage of customers with >100,000 additional duplicates: {threshold_percentage:.1f}%")
print(f"Percentage of customers with ≤100,000 additional duplicates: {100 - threshold_percentage:.1f}%")

# Also show top 20 customers with most duplicates
print("\nTop 20 customers with most additional Steps duplicates:")
customer_steps_additional_top20 = customer_steps_additional.head(20)
print(customer_steps_additional_top20)

plt.xlim(-1, len(customer_steps_additional)/2)  # Set x-axis limits to cover all customers
plt.show()

# Print customer names with >100,000 additional duplicates for easy copying
customers_steps_over_threshold = customer_steps_additional[customer_steps_additional > 100000].index.tolist()
print(f"Customers with >100,000 additional Steps duplicates ({len(customers_steps_over_threshold)} total):")
print(customers_steps_over_threshold)

In [ ]:
# Remove duplicates by taking the average where there are multiple entries per timestamp


df_steps_avg = df_steps_expanded_groupby.agg({
    "SPM": "mean",
    "for_id": "first",
    "timezoneOffset": "first",
    # "startTimestamp": "first",
    # "endTimestamp": "max",
    # "local_startTimestamp": "first",
    # "local_endTimestamp": "max",
}).reset_index()

df_steps_min = df_steps_expanded_groupby.agg(
    {
        "SPM": "min",
    }
).reset_index()

df_steps_max = df_steps_expanded_groupby.agg(
    {
        "SPM": "max",
    }
).reset_index()

df_steps_avg

In [ ]:
diff_steps = df_steps_max["SPM"] - df_steps_min["SPM"]
print(f"Proportion of repeated Steps data that have different values: {(diff_steps>0).sum() / (len(df_steps_expanded) - len(df_steps_avg)):.2%}")
print("Out of this, the following are the statistics of the differences:")
diff_steps[diff_steps > 0].describe([0.25,0.5,0.75,0.9, 0.95, 0.99])

## Steps Aggregations
at this point we've got nice df_steps_avg, so we finally can perform aggregations

SPM is just cadence in other words

there's a problem distinguishing the zero steps and missing data

In [ ]:
df_steps = df_steps_avg
df_steps

In [ ]:
df_steps['t_day'] = df_steps['timestamp'].dt.floor('d')
df_steps['t_hour'] = df_steps['timestamp'].dt.floor('h')
# ! isnighttime depends on the timezone, preferably this should be in local time
# TODO decide how much timezone changes is the issue for this, most of the time subjects are in Germany
# TODO DST? 
df_steps["local_timestamp"] = (df_steps["timestamp"] + pd.to_timedelta(df_steps["timezoneOffset"], unit="min")).dt.tz_localize(None)
df_steps["isnighttime"] = df_steps['local_timestamp'].dt.hour < 6 # 00:00 - 05:59
df_steps

In [ ]:
# Create daily steps aggregates
df_steps_daily = df_steps.groupby(['customer', 't_day']).agg(
    for_id=('for_id', 'first'),
    StepsInDay=('SPM', 'sum'),
    # TODO change it for .apply and make all the quantiles and max computed in one run
    SPM_25pct=("SPM", lambda x: x.quantile(0.25)),
    SPM_50pct=("SPM", lambda x: x.quantile(0.50)),
    SPM_75pct=("SPM", lambda x: x.quantile(0.75)),
    SPM_max=('SPM', 'max'),
    SPM_count=('SPM', 'count'), # total number of minues for which we've got the data
    SPM_mean=('SPM', 'mean'), # it's not the average steps/min in the day, for that we would need to fill data with 0s; use StepsInDay/(24*60) to get that value
    SPM_std=('SPM', 'std'),
    SPM_skew=('SPM', "skew"),
    SPM_kurtosis=('SPM', lambda x: x.kurtosis()),
    # SPM_min=('SPM', 'min'), # doesn't make much sense in Steps (0)
    StepsAtNight_sum=('SPM', lambda x: x[df_steps.loc[x.index, 'isnighttime']].sum()),
    StepsAtNight_mean=('SPM', lambda x: x[df_steps.loc[x.index, 'isnighttime']].mean()),
).reset_index()
df_steps_daily["StepsAtNight_mean"] = df_steps_daily["StepsAtNight_mean"].fillna(0)

df_steps_daily

In [ ]:
# Create hourly steps aggregates
df_steps_hourly = df_steps.groupby(['customer', 't_hour']).agg(
    for_id=('for_id', 'first'),
    Steps_inHour=('SPM', 'sum'),
    SPM_max_inHour=('SPM', 'max'),
    SPM_count_inHour=('SPM', 'count'),
    SPM_mean_inHour=('SPM', 'mean'),
    SPM_std_inHour=('SPM', 'std'),
    SPM_skew_inHour=('SPM', "skew"),
    SPM_kurtosis_inHour=('SPM', lambda x: x.kurtosis()),
    # SPM_min=('SPM', 'min'),
).reset_index()

df_steps_hourly["t_day"] = df_steps_hourly["t_hour"].dt.floor("d")
df_steps_hourly["clock_hour"] = df_steps_hourly["t_hour"].dt.hour

df_steps_hourly

In [ ]:
df_steps_daily = df_steps_daily.merge(
    df_steps_hourly.groupby(["customer", "t_day"]).agg(
        StepsPerHour=("Steps_inHour", "mean"),
        SPM_max_avgbyHour=("SPM_max_inHour", "mean"),
        SPM_mean_avgbyHour=("SPM_mean_inHour", "mean"),
        SPM_std_avgbyHour=("SPM_std_inHour", "mean"),
        SPM_skew_avgbyHour=("SPM_skew_inHour", "mean"),
        SPM_kurtosis_avgbyHour=("SPM_kurtosis_inHour", "mean"),
    ),
    on=["customer", "t_day"],
)

In [ ]:
# Most Active hour
df_steps_daily = df_steps_daily.merge(
    df_steps_hourly.loc[
        df_steps_hourly.groupby(["customer", "t_day"])["Steps_inHour"].idxmax()
    ][["customer", "t_day", "Steps_inHour", "clock_hour", "SPM_max_inHour", "SPM_mean_inHour", ]].rename(
        columns={
            "Steps_inHour": "Steps_in_most_active_hour",
            "clock_hour": "most_active_hour",  # hour with most steps
            "SPM_max_inHour": "max_spm_in_most_active_hour",
            "SPM_mean_inHour": "avg_spm_in_most_active_hour",
        }
    ),
    on=["customer", "t_day"],
)

In [ ]:
df_steps_hourly["Steps_inHour_cumsum"] = df_steps_hourly.groupby(["customer", "t_day"])["Steps_inHour"].cumsum()
    
def percentile_hours(group, percentiles=(0.25, 0.5, 0.75)):
    total_steps = group["Steps_inHour_cumsum"].iloc[-1]
    result = {}
    for p in percentiles:
        idx = group["Steps_inHour_cumsum"].searchsorted(p * total_steps)
        result[f"hour_reach_{int(p*100)}pct_Steps_cumsum"] = group.iloc[idx]["t_hour"].hour
    return pd.Series(result)


df_steps_daily = df_steps_daily.merge(
    df_steps_hourly.groupby(["customer", "t_day"]).apply(percentile_hours, include_groups=False),
    on=["customer", "t_day"],
)

In [ ]:
# TODO change to local_time for night calculation
df_steps_daily.columns

inspired by RADAR MDD

| Column Name | Description |
|-------------|-------------|
| customer | Customer/participant identifier |
| t_day | Day timestamp (floor to day) UTC |
| for_id | Record identifier |
| StepsInDay | Total number of walked steps within the day |
| SPM_25pct | 25th percentile of daily steps per minute distribution |
| SPM_50pct | 50th percentile of daily steps per minute distribution |
| SPM_75pct | 75th percentile of daily steps per minute distribution |
| SPM_max | Maximum steps per minute along all day |
| SPM_count | Number of minutes with step data available |
| SPM_mean | Mean steps per minute along all day (among available records) |
| SPM_std | Standard deviation of steps per minute along all day |
| SPM_skew | Skewness of steps per minute along all day |
| SPM_kurtosis | Kurtosis of steps per minute along all day |
| StepsAtNight_sum | Sum of steps per minute during nighttime (00:00-05:59) |
| StepsAtNight_mean | Mean steps per minute during nighttime (00:00-05:59) |
| StepsPerHour | Mean of hourly step sums (sum of steps per minute, averaged by hour) |
| SPM_max_avgbyHour | Maximum steps per minute, averaged by hour |
| SPM_mean_avgbyHour | Mean steps per minute, averaged by hour |
| SPM_std_avgbyHour | Standard deviation of steps per minute, averaged by hour |
| SPM_skew_avgbyHour | Skewness of steps per minute, averaged by hour |
| SPM_kurtosis_avgbyHour | Kurtosis of steps per minute, averaged by hour |
| Steps_in_most_active_hour | Maximum of the hourly sum of steps along all day |
| most_active_hour | Most active hour (hour with maximum hourly sum of steps) |
| max_spm_in_most_active_hour | Maximum step cadence during the most active hour |
| avg_spm_in_most_active_hour | Average step cadence during the most active hour |
| hour_reach_25pct_Steps_cumsum | Hour at which 25th percentile of daily steps occurred (cumulative) |
| hour_reach_50pct_Steps_cumsum | Hour at which 50th percentile of daily steps occurred (cumulative) |
| hour_reach_75pct_Steps_cumsum | Hour at which 75th percentile of daily steps occurred

| No. | Column Name | Description |
|-----|-------------|-------------|
|  1  | `id` | Unique identifier wearable and ema data within subproject 6 (SP6) |
|  2  |`for_id` | Unique identifier across all PREACT subprojects and redcap |
|  3  |`date` | Day timestamp (floor to day) UTC |
|  4  | `n_steps_day` | Total number of walked steps within the day |
|  5  | `spm_25_steps` | 25th percentile of daily steps per minute distribution |
|  6  | `spm_50_steps` | 50th percentile of daily steps per minute distribution |
|  7  | `spm_75_steps` | 75th percentile of daily steps per minute distribution |
|  8  | `spm_max_steps` | Maximum steps per minute along all day |
|  9  | `spm_count_steps` | Number of minutes with step data available |
|  10 | `spm_mean_steps` | Mean steps per minute along all day (among available records) |
|  11 | `spm_std_steps` | Standard deviation of steps per minute along all day |
|  12 | `spm_skew_steps` | Skewness of steps per minute along all day |
|  13 | `spm_kurtosis_steps` | Kurtosis of steps per minute along all day |
|  14 | `night_sum_steps` | Sum of steps per minute during nighttime (00:00-05:59) |
|  15 | `night_mean_steps` | Mean steps per minute during nighttime (00:00-05:59) |
|  16 | `n_hour_steps` | Mean of hourly step sums (sum of steps per minute, averaged by hour) |
|  17 | `spm_max_avghr_steps` | Maximum steps per minute, averaged by hour |
|  18 | `spm_mean_avghr_steps` | Mean steps per minute, averaged by hour |
|  19 | `spm_std_avghr_steps` | Standard deviation of steps per minute, averaged by hour |
|  20 | `spm_skew_avghr_steps` | Skewness of steps per minute, averaged by hour |
|  21 | `spm_kurtosis_avghr_steps` | Kurtosis of steps per minute, averaged by hour |
|  22 | `n_steps_activehr_steps` | Maximum of the hourly sum of steps along all day |
|  23 | `timestamp_max_activehr_steps` | Most active hour (hour with maximum hourly sum of steps) |
|  24 | `max_spm_activehr_steps` | Maximum step cadence during the most active hour |
|  25 | `mean_spm_activehr_steps` | Average step cadence during the most active hour |
|  26 | `dailysteps_25perc_steps` | Hour at which 25th percentile of daily steps occurred (cumulative) |
|  27 | `dailysteps_50perc_steps` | Hour at which 50th percentile of daily steps occurred (cumulative) |
|  28 | `dailysteps_75perc_steps` | Hour at which 75th percentile of daily steps occurred

#### Example plot

In [ ]:
# example plot
df_steps_plot = df_steps_avg[(df_steps_avg["customer"] == "05kz") & (df_steps_avg["timestamp"] >= pd.to_datetime("2023-10-19")) 
                 & (df_steps_avg["timestamp"] < pd.to_datetime("2023-10-22"))]
df_steps_plot

plt.figure(figsize=(12, 6))
plt.scatter(df_steps_plot['timestamp'], df_steps_plot['SPM'], linewidth=0.5) # scatter vs line plot
plt.title(f'Steps Data for Customer {df_steps_plot["customer"].iloc[0]} on {df_steps_plot["timestamp"].dt.date.iloc[0]} onwards')
plt.xlabel('Time')
plt.ylabel('Steps per Second')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ActivityType
how are these encoded?
-> google sheet

In [ ]:
df_backup[df_backup["type"] == "ActivityType"]["longValue"].describe([0.8,0.9,0.99, 0.999])
plot_cdf(df_backup[df_backup["type"] == "ActivityType"]["longValue"])

In [ ]:
df_backup[df_backup["type"] == "ActivityTypeDetail1"]["longValue"].describe([0.8,0.9,0.99, 0.999])


In [ ]:
df_backup[df_backup["type"] == "ActivityTypeDetail2"]["longValue"].describe([0.75, 0.9, 0.99, 0.999])


# ~ArtialFibrillation~ -> will not use

In [ ]:
df_artialfibrillation = df_backup[df_backup["type"] == "AtrialFibrillationDetection"]


In [ ]:

df_artialfibrillation.groupby(["customer"])["booleanValue"].mean().reset_index()["booleanValue"].astype(float).describe([0.1,0.25,0.5,0.75,0.9])
plot_cdf(df_artialfibrillation.groupby(["customer"])["booleanValue"].mean().reset_index()["booleanValue"].astype(float))

# WalkBinary and Steps comparison
no rocket science here, this is the most important result:

As percentage of records:

Steps>0, but WalkBinary=False 84.35%

Steps>0, and WalkBinary=True  13.46%

Steps=0, but WalkBinary=True   2.19%


The threshold for WalkBinary=True seems to be ~60 SPM, but there are other factors, so that WalkBinary doesn't go to False on the first minute with 0 SPM

In [ ]:
df_backup[df_backup["type"] == "Steps"]["customer"].nunique()

In [ ]:
df_bup_walkbinary = df_backup[df_backup["type"] == "WalkBinary"]
df_bup_walkbinary["customer"].nunique()

In [ ]:
df_bup_walkbinary["booleanValue"].unique()

In [ ]:
df_bup_walkbinary["startTimestamp"].dt.second.describe([0.8, 0.9, 0.95, 0.99, 0.999])

In [ ]:
df_walkbinary_original = df_backup[(df_backup["type"] == "WalkBinary")]
df_walkbinary_original = df_walkbinary_original[["customer", "startTimestamp", "endTimestamp", "booleanValue"]]
df_walkbinary_original = df_walkbinary_original.rename(columns={"booleanValue": "WalkBinary"})
df_walkbinary_original

In [ ]:
print(f"{(df_walkbinary_original['startTimestamp'].dt.second != 0).sum() / len(df_walkbinary_original):.2%}")

bins = np.arange(0, 61, 1) - 0.5  # 0 to 59 seconds
df_walkbinary_original["startTimestamp"].dt.second.describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99])
plt.figure(figsize=(10, 6))
plt.hist(df_walkbinary_original["startTimestamp"].dt.second, bins=bins, alpha=0.5, edgecolor='black', label='Start')
plt.hist(df_walkbinary_original["endTimestamp"].dt.second, bins=bins, alpha=0.5, edgecolor='black', label='End')
plt.title('Distribution of Start/End Timestamp Seconds for WalkBinary Data')
plt.xlabel('Second (0-59)')
plt.ylabel('Frequency')
plt.yscale('log')  # Use logarithmic scale for better visibility
plt.grid(True, alpha=0.3)
plt.xticks(np.arange(0, 60, 5))
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_walkbinary_original["duration"] = (df_walkbinary_original["endTimestamp"] - df_walkbinary_original["startTimestamp"]).dt.total_seconds().fillna(0).astype(int)
df_walkbinary_original["duration"].describe([0.003, 0.004, 0.005, 0.0075, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.993, 0.994, 0.995, 0.9975, 0.999])

mask_under_59 = df_walkbinary_original["duration"] < 59
mask_over_61 = df_walkbinary_original["duration"] > 61
print(f"Percentage of WalkBinary samples with duration not in [59, 61]: {((mask_under_59 | mask_over_61).sum() / len(df_walkbinary_original)) * 100:.2f}%")

In [ ]:
df_walkbinary_original["startTimestamp_minute"] = df_walkbinary_original["startTimestamp"].dt.round('min')
df_walkbinary_original["duration_minutes"] = np.maximum(1, (df_walkbinary_original["duration"] / 60).round(0).astype(int))
print(f"Minimum duration in minutes: {df_walkbinary_original['duration_minutes'].min()}")

# Expand the dataset
df_walkbinary_expanded = df_walkbinary_original.loc[df_walkbinary_original.index.repeat(df_walkbinary_original['duration_minutes'])].copy()
df_walkbinary_expanded['time_offset'] = df_walkbinary_expanded.groupby(level=0).cumcount()
df_walkbinary_expanded['timestamp'] = df_walkbinary_expanded['startTimestamp'] + pd.to_timedelta(df_walkbinary_expanded['time_offset'], unit='min')
df_walkbinary_expanded

In [ ]:
# Check for overlapping entries in WalkBinary data
df_walkbinary_expanded_groupby = df_walkbinary_expanded.groupby(['customer','timestamp'])

df_walkbinary_expanded_groupby.size().value_counts().sort_index()

In [ ]:
df_walkbinary_size = df_walkbinary_expanded_groupby.size().reset_index(name='n_repeat')
# drop all the rows where count is 1
df_walkbinary_size = df_walkbinary_size[df_walkbinary_size['n_repeat'] > 1]
df_walkbinary_size.sort_values(by='n_repeat', ascending=False)

df_walkbinary_size["n_repeat"].describe([0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99])

In [ ]:
df_walkbinary_size["n_additional"] = df_walkbinary_size["n_repeat"] - 1
df_walkbinary_size.groupby("customer")["n_additional"].sum().sort_values(ascending=False)
customer_walkbinary_additional = df_walkbinary_size.groupby("customer")["n_additional"].sum().sort_values(ascending=False)
# Get all customers from df_backup and ensure they're included with 0 if not in customer_walkbinary_additional
all_customers = df_backup["customer"].unique()
customer_walkbinary_additional = customer_walkbinary_additional.reindex(all_customers, fill_value=0).sort_values(ascending=False)

print(customer_walkbinary_additional.describe([]).round(2))
customer_walkbinary_additional.quantile([0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.96, 0.97, 0.98, 0.99, 1.0], interpolation="linear").round(2)

In [ ]:
plt.figure(figsize=(12, 8))
plt.bar(range(len(customer_walkbinary_additional)), customer_walkbinary_additional.values)

# Create percentage-based x-axis labels (reversed from 100% to 0%)
n_customers = len(customer_walkbinary_additional)
percentage_ticks = np.arange(0, n_customers, max(1, n_customers // 20))  # Show ~20 ticks
percentage_labels = [f"{100 - (i/n_customers)*100:.0f}%" for i in percentage_ticks]

plt.xticks(percentage_ticks, percentage_labels)
plt.xlabel('Patients Percentile (100% = highest duplicates, 0% = lowest duplicates)')
plt.ylabel('Number of Additional Duplicate WalkBinary Measurements (minutes)')
plt.title('Additional Duplicate WalkBinary Measurements by Customer')
plt.grid(True, alpha=0.3)

# Find the position where customer_walkbinary_additional crosses 10,000 (lower threshold for WalkBinary)
threshold_value = 10000
threshold_index = (customer_walkbinary_additional > threshold_value).sum()
threshold_percentage = 100 - (threshold_index / n_customers) * 100

# Add horizontal line at threshold
plt.axhline(y=threshold_value, color='red', linestyle='--', linewidth=2, label=f'{threshold_value:,} threshold')

# Add vertical line at the threshold position
plt.axvline(x=threshold_index, color='red', linestyle='--', linewidth=2, alpha=0.7)

# Add text annotation
plt.text(threshold_index + 20, threshold_value * 1.5, f'{threshold_percentage:.1f}% of patients\nhave <{threshold_value:,} duplicates each', 
         bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.legend()
plt.tight_layout()

print(f"WalkBinary Threshold analysis:")
print(f"Number of customers with >{threshold_value:,} additional duplicates: {threshold_index}")
print(f"Percentage of customers with >{threshold_value:,} additional duplicates: {threshold_percentage:.1f}%")
print(f"Percentage of customers with ≤{threshold_value:,} additional duplicates: {100 - threshold_percentage:.1f}%")

# Also show top 20 customers with most duplicates
print("\nTop 20 customers with most additional WalkBinary duplicates:")
customer_walkbinary_additional_top20 = customer_walkbinary_additional.head(20)
print(customer_walkbinary_additional_top20)

plt.xlim(-1, len(customer_walkbinary_additional)/2)  # Set x-axis limits to cover all customers
plt.show()

# Print customer names with >threshold additional duplicates for easy copying
customers_walkbinary_over_threshold = customer_walkbinary_additional[customer_walkbinary_additional > threshold_value].index.tolist()
print(f"Customers with >{threshold_value:,} additional WalkBinary duplicates ({len(customers_walkbinary_over_threshold)} total):")
print(customers_walkbinary_over_threshold)

In [ ]:
# Remove duplicates - for boolean data, we need to decide how to handle conflicts
# Let's check if there are conflicts (different boolean values for same timestamp/customer)

df_walkbinary_true = df_walkbinary_expanded_groupby.agg({
    'WalkBinary': lambda x: x.any(),  # True if any value is True
}).reset_index()

df_walkbinary_false = df_walkbinary_expanded_groupby.agg({
    'WalkBinary': lambda x: x.all(),  # True only if all values are True
}).reset_index()

df_walkbinary_count = df_walkbinary_expanded_groupby.agg({
    'WalkBinary': 'count',
}).reset_index()

# Check for conflicts: cases where we have both True and False values for same timestamp
conflicts = df_walkbinary_true.merge(df_walkbinary_false, on=['customer', 'timestamp'], suffixes=('_any', '_all'))
conflicts['has_conflict'] = (conflicts['WalkBinary_any'] != conflicts['WalkBinary_all'])

print(f"Number of timestamp conflicts (mix of True/False): {conflicts['has_conflict'].sum()}")
print(f"Proportion of conflicts out of duplicated timestamps: {conflicts['has_conflict'].sum() / len(conflicts):.4%}")

# Use the "any" approach (if any reading says walking, consider it walking)
df_walkbinary_processed = df_walkbinary_true.rename(columns={'WalkBinary': 'WalkBinary'})
df_walkbinary_processed

## WalkBinary and Steps comparison

In [ ]:
# Merge WalkBinary and Steps data for comparison
# Both datasets are at minute-level resolution with customer and timestamp as keys

df_comparison = df_walkbinary_processed.merge(
    df_steps_avg[['customer', 'timestamp', 'SPM']], 
    on=['customer', 'timestamp'], 
    how='outer',
    indicator=True
)

print("Merge results:")
print(df_comparison['_merge'].value_counts())
print(f"Total records: {len(df_comparison)}")

# Fill missing values
df_comparison['WalkBinary'] = df_comparison['WalkBinary'].fillna(False)
df_comparison['SPM'] = df_comparison['SPM'].fillna(0.0)

df_comparison

In [ ]:
print("As percentage of records:")
merge_value_counts = df_comparison['_merge'].value_counts()
print("Steps>0, but WalkBinary=False",f"{merge_value_counts['right_only'] / merge_value_counts.sum():.2%}")
print("Steps>0, and WalkBinary=True",f"{merge_value_counts['both'] / merge_value_counts.sum():.2%}")
print("Steps=0, but WalkBinary=True",f"{merge_value_counts['left_only'] / merge_value_counts.sum():.2%}")

### run cells below only if you really need nice plot
(llm created, but seems legit, at least matches the distribution from above)

In [ ]:
# Create categories for analysis
df_comparison['has_steps'] = df_comparison['SPM'] > 0
df_comparison['walking_detected'] = df_comparison['WalkBinary'] == True

# Create comparison categories
def categorize_walking_steps(row):
    if row['walking_detected'] and row['has_steps']:
        return 'Walking_True_Steps_Yes'
    elif row['walking_detected'] and not row['has_steps']:
        return 'Walking_True_Steps_No'
    elif not row['walking_detected'] and row['has_steps']:
        return 'Walking_False_Steps_Yes'
    else:
        return 'Walking_False_Steps_No'

df_comparison['category'] = df_comparison.apply(categorize_walking_steps, axis=1)

# Summary of categories
category_summary = df_comparison['category'].value_counts()
category_summary_pct = df_comparison['category'].value_counts(normalize=True) * 100

print("Category Summary:")
for cat, count in category_summary.items():
    pct = category_summary_pct[cat]
    print(f"{cat}: {count:,} ({pct:.2f}%)")

category_summary

In [ ]:
# Analyze specific cases of interest
print("=== CASE 1: WalkBinary=True but SPM=0 (Walking detected but no steps) ===")
walking_no_steps = df_comparison[df_comparison['category'] == 'Walking_True_Steps_No']
print(f"Total occurrences: {len(walking_no_steps):,}")
print(f"Unique customers affected: {walking_no_steps['customer'].nunique()}")
print(f"Percentage of all walking detections: {len(walking_no_steps) / df_comparison[df_comparison['walking_detected']].shape[0] * 100:.2f}%")

# Show some examples
print("\nExamples:")
print(walking_no_steps[['customer', 'timestamp', 'WalkBinary', 'SPM']].head(10))

In [ ]:
print("\n=== CASE 2: WalkBinary=False but SPM>0 (Steps detected but no walking) ===")
steps_no_walking = df_comparison[df_comparison['category'] == 'Walking_False_Steps_Yes']
print(f"Total occurrences: {len(steps_no_walking):,}")
print(f"Unique customers affected: {steps_no_walking['customer'].nunique()}")
print(f"Percentage of all step detections: {len(steps_no_walking) / df_comparison[df_comparison['has_steps']].shape[0] * 100:.2f}%")

# Analyze SPM distribution for this case
print(f"\nSPM statistics for steps without walking detection:")
print(steps_no_walking['SPM'].describe([0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

# Show some examples
print("\nExamples:")
print(steps_no_walking[['customer', 'timestamp', 'WalkBinary', 'SPM']].head(10))

In [ ]:
print("\n=== CASE 3: WalkBinary=True and SPM>0 (Both agree on walking) ===")
walking_with_steps = df_comparison[df_comparison['category'] == 'Walking_True_Steps_Yes']
print(f"Total occurrences: {len(walking_with_steps):,}")
print(f"Unique customers: {walking_with_steps['customer'].nunique()}")

# Analyze SPM distribution when both agree
print(f"\nSPM statistics when both walking detection and steps agree:")
print(walking_with_steps['SPM'].describe([0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

print("\nExamples:")
print(walking_with_steps[['customer', 'timestamp', 'WalkBinary', 'SPM']].head(10))

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Overall category distribution
axes[0,0].pie(category_summary.values, labels=category_summary.index, autopct='%1.1f%%')
axes[0,0].set_title('Distribution of WalkBinary vs Steps Categories')

# 2. SPM distribution by walking detection
walking_true = df_comparison[df_comparison['walking_detected']]['SPM']
walking_false = df_comparison[~df_comparison['walking_detected']]['SPM']

axes[0,1].hist(walking_true, bins=50, alpha=0.7, label='WalkBinary=True', density=True)
axes[0,1].hist(walking_false, bins=50, alpha=0.7, label='WalkBinary=False', density=True)
axes[0,1].set_xlabel('Steps Per Minute (SPM)')
axes[0,1].set_ylabel('Density')
axes[0,1].set_title('SPM Distribution by Walking Detection')
axes[0,1].legend()
axes[0,1].set_xlim(0, 200)  # Focus on reasonable SPM range

# 3. Boxplot of SPM by category
category_data = [df_comparison[df_comparison['category'] == cat]['SPM'].values 
                for cat in category_summary.index]
axes[1,0].boxplot(category_data, labels=[cat.replace('_', '\n') for cat in category_summary.index])
axes[1,0].set_ylabel('Steps Per Minute (SPM)')
axes[1,0].set_title('SPM Distribution by Category')
axes[1,0].tick_params(axis='x', rotation=45)

# 4. Scatter plot: SPM vs WalkBinary (with jitter for visibility)
walking_points = df_comparison[df_comparison['walking_detected']]
no_walking_points = df_comparison[~df_comparison['walking_detected']]

# Add jitter to binary variable for visibility
jitter_strength = 0.1
walking_jitter = np.random.normal(1, jitter_strength, len(walking_points))
no_walking_jitter = np.random.normal(0, jitter_strength, len(no_walking_points))

axes[1,1].scatter(walking_jitter, walking_points['SPM'], alpha=0.1, s=1, label='WalkBinary=True')
axes[1,1].scatter(no_walking_jitter, no_walking_points['SPM'], alpha=0.1, s=1, label='WalkBinary=False')
axes[1,1].set_xlabel('WalkBinary (with jitter)')
axes[1,1].set_ylabel('Steps Per Minute (SPM)')
axes[1,1].set_title('SPM vs WalkBinary Scatter Plot')
axes[1,1].set_xticks([0, 1])
axes[1,1].set_xticklabels(['False', 'True'])
axes[1,1].legend()
axes[1,1].set_ylim(0, 200)  # Focus on reasonable SPM range

plt.tight_layout()
plt.show()

In [ ]:
# Analyze patterns by customer
customer_analysis = df_comparison.groupby('customer').agg({
    'category': lambda x: (x == 'Walking_True_Steps_No').sum(),  # Walking detected but no steps
    'WalkBinary': lambda x: (x == True).sum(),  # Total walking detections
    'has_steps': lambda x: x.sum(),  # Total step detections
}).rename(columns={
    'category': 'walking_no_steps_count',
    'WalkBinary': 'total_walking_detections',
    'has_steps': 'total_step_detections'
})

# Add more analysis
customer_analysis['steps_no_walking_count'] = df_comparison.groupby('customer')['category'].apply(
    lambda x: (x == 'Walking_False_Steps_Yes').sum()
)

customer_analysis['both_agree_count'] = df_comparison.groupby('customer')['category'].apply(
    lambda x: (x == 'Walking_True_Steps_Yes').sum()
)

customer_analysis['total_records'] = df_comparison.groupby('customer').size()

# Calculate rates
customer_analysis['walking_no_steps_rate'] = customer_analysis['walking_no_steps_count'] / customer_analysis['total_walking_detections']
customer_analysis['steps_no_walking_rate'] = customer_analysis['steps_no_walking_count'] / customer_analysis['total_step_detections']

# Fill NaN values (division by zero)
customer_analysis = customer_analysis.fillna(0)

print("Customer Analysis Summary:")
print(customer_analysis.describe())

# Show customers with highest discrepancy rates
print("\nTop 10 customers with highest 'walking detected but no steps' rate:")
top_walking_no_steps = customer_analysis.nlargest(10, 'walking_no_steps_rate')[
    ['walking_no_steps_count', 'total_walking_detections', 'walking_no_steps_rate']
]
print(top_walking_no_steps)

print("\nTop 10 customers with highest 'steps but no walking detected' rate:")
top_steps_no_walking = customer_analysis.nlargest(10, 'steps_no_walking_rate')[
    ['steps_no_walking_count', 'total_step_detections', 'steps_no_walking_rate']
]
print(top_steps_no_walking)

In [ ]:
# Detailed analysis for a specific customer
# Let's pick a customer with interesting patterns
if len(customer_analysis) > 0:
    # Pick customer with moderate activity and some discrepancies
    interesting_customers = customer_analysis[
        (customer_analysis['total_walking_detections'] > 100) & 
        (customer_analysis['total_step_detections'] > 100) &
        (customer_analysis['walking_no_steps_rate'] > 0.1)
    ]
    
    if len(interesting_customers) > 0:
        example_customer = interesting_customers.index[0]
        print(f"\n=== DETAILED ANALYSIS FOR CUSTOMER: {example_customer} ===")
        
        customer_data = df_comparison[df_comparison['customer'] == example_customer].copy()
        customer_data['date'] = customer_data['timestamp'].dt.date
        
        print(f"Total records: {len(customer_data)}")
        print(f"Date range: {customer_data['date'].min()} to {customer_data['date'].max()}")
        print(f"Categories breakdown:")
        print(customer_data['category'].value_counts())
        
        # Show a few days of data
        sample_date = customer_data['date'].value_counts().index[0]  # Pick date with most records
        sample_data = customer_data[customer_data['date'] == sample_date].sort_values('timestamp')
        
        print(f"\nSample data for {sample_date} (first 20 records):")
        print(sample_data[['timestamp', 'WalkBinary', 'SPM', 'category']].head(20))
        
        # Plot timeline for this customer
        plt.figure(figsize=(15, 8))
        
        # Create subplots
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10), sharex=True)
        
        # Plot 1: WalkBinary over time
        walking_times = sample_data[sample_data['WalkBinary']]['timestamp']
        ax1.scatter(walking_times, [1]*len(walking_times), alpha=0.6, s=20, color='blue', label='Walking Detected')
        ax1.set_ylabel('WalkBinary')
        ax1.set_ylim(-0.1, 1.1)
        ax1.set_title(f'Walking Detection and Steps for Customer {example_customer} on {sample_date}')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: SPM over time
        ax2.scatter(sample_data['timestamp'], sample_data['SPM'], alpha=0.6, s=10, color='red')
        ax2.set_ylabel('Steps Per Minute')
        ax2.set_xlabel('Time')
        ax2.set_title('Steps Per Minute')
        ax2.grid(True, alpha=0.3)
        
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("\nNo customers found with the specified criteria for detailed analysis")
else:
    print("\nNo customer data available for analysis")

# Sleep

## Sleep overlap analysis

In [ ]:
df_bup_sleep = df_backup[df_backup["type"].isin([
    "SleepDeepBinary",
    "SleepLightBinary",
    # "SleepREMBinary", # only 10 people got these records
    # "SnoringBinary", # only 2 people got these records #? maybe relevant for these 2 people?
    "SleepStateBinary",
    "SleepBinary",
    "SleepInBedBinary",
    "SleepAwakeBinary"
])]

In [ ]:
df_bup_sleep

In [ ]:
df_bup_sleep.groupby("type", observed=True)["customer"].nunique().sort_values(ascending=False)

In [ ]:
df_bup_sleep = df_bup_sleep.sort_values(by=["customer", "local_start_time", "start_end"], ascending=[True, True, False]).reset_index()

In [ ]:
df_bup_sleep["next_customer"] = df_bup_sleep["customer"].shift(-1)
df_bup_sleep["to_next_start"] = df_bup_sleep["local_start_time"].diff().shift(-1)
df_bup_sleep["next_type"] = df_bup_sleep["type"].shift(-1)

customer_change = df_bup_sleep["customer"] != df_bup_sleep["next_customer"]
df_bup_sleep[customer_change]["to_next_start"] = pd.NA
df_bup_sleep[customer_change]["next_type"] = pd.NA

In [ ]:
wrong_mask = (
    (df_bup_sleep["to_next_start"].dt.total_seconds() < df_bup_sleep["start_end"])
    & (df_bup_sleep["next_customer"] == df_bup_sleep["customer"])
    & (df_bup_sleep["next_type"] != df_bup_sleep["type"])
)
wrong_indices = df_bup_sleep.index[wrong_mask]
# df_bup_sleep[df_bup_sleep["to_next_start"].dt.total_seconds() < df_bup_sleep["duration"]]
# df_bup_sleep.where(wrong_mask)
wrong_and_next_indices = np.sort(np.concatenate([wrong_indices, wrong_indices + 1]))
print(df_bup_sleep.iloc[wrong_and_next_indices]["customer"].value_counts())
print(df_bup_sleep.iloc[wrong_and_next_indices].groupby("customer")["local_start_time_day"].value_counts().sort_index())
df_bup_sleep.iloc[wrong_and_next_indices]

In [ ]:
df_wrong_xqjb["type"].unique()

In [ ]:
# Filter the wrong_and_next_indices for customer 'Xqjb'
df_wrong_xqjb = df_bup_sleep.iloc[wrong_and_next_indices]
df_wrong_xqjb = df_wrong_xqjb[df_wrong_xqjb["customer"] == "Xqjb"]

# Plot intervals as lines, with red for start and blue for end, spread within each sleep type level
plt.figure(figsize=(12, 6))
y_offset_step = 0.05  # Spread within each level

# Assign a numeric y position for each sleep type
unique_types = df_wrong_xqjb["type"].unique()
# type_to_y = {"SleepDeepBinary": 0, "SleepLightBinary": 1, "SleepAwakeBinary": 2}
type_to_y = {"SleepDeepBinary": 0, "SleepLightBinary": 1, "SleepAwakeBinary": 2, "SleepInBedBinary": 3, "SleepStateBinary": 4, "SleepBinary": 5}
unique_types = list(type_to_y.keys())

# Track how many intervals have been plotted for each type
type_counts = {t: 0 for t in unique_types}

for idx, row in df_wrong_xqjb.iterrows():
    base_y = type_to_y[row["type"]]
    offset = type_counts[row["type"]]
    y_plot = base_y + offset * y_offset_step
    type_counts[row["type"]] += 1

    # Draw the interval as a line
    plt.plot([row["local_start_time"], row["local_end_time"]], [y_plot, y_plot], color='purple', linewidth=2)
    # Mark the start (red) and end (blue)
    plt.scatter(row["local_start_time"], y_plot, color='red', s=60, label='Start' if idx == 0 else "")
    plt.scatter(row["local_end_time"], y_plot, color='blue', s=60, label='End' if idx == 0 else "")
    # Vertical dashed lines at start and end
    plt.axvline(row["local_start_time"], color='red', linestyle='dashed', linewidth=1, alpha=0.4)
    plt.axvline(row["local_end_time"], color='blue', linestyle='dashed', linewidth=1, alpha=0.4)
    # Optionally, annotate with type
    # plt.text(row["startTimestamp"], y_plot + 0.05, row["type"], fontsize=8, va='bottom')

plt.title("Wrong Sleep Intervals for Customer Xqjb (Spread within Level)")
plt.xlabel("Timestamp")
plt.yticks([type_to_y[t] for t in unique_types], unique_types)
plt.ylabel("Sleep Type")
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()

## SleepBinary vs SleepStateBinary

In [ ]:
df_bup_sleep = df_backup[df_backup["type"].isin([
    "SleepDeepBinary",
    "SleepLightBinary",
    # "SleepREMBinary", # only 4 people got these records
    # "SnoringBinary", # only 2 people got these records #? maybe relevant for these 2 people?
    "SleepStateBinary",
    "SleepBinary",
    "SleepInBedBinary",
    "SleepAwakeBinary"
])]

df_bup_sleep

In [ ]:
print("Unique customers in sleep data:", df_bup_sleep["customer"].nunique())
df_bup_sleep.groupby("type")["customer"].nunique().sort_values(ascending=False)

In [ ]:
df_SleepStateBin = df_bup_sleep[df_bup_sleep["type"] == "SleepStateBinary"]
df_SleepBin = df_bup_sleep[df_bup_sleep["type"] == "SleepBinary"]

In [ ]:
# Get the monthly periods for both datasets
sleepstate_months = df_SleepStateBin["local_start_time"].dt.to_period("M")
sleep_months = df_SleepBin["local_start_time"].dt.to_period("M")

# Create histogram comparing the two
plt.figure(figsize=(15, 8))

# Convert periods to strings for plotting
sleepstate_month_counts = sleepstate_months.value_counts().sort_index()
sleep_month_counts = sleep_months.value_counts().sort_index()

# Get all unique months across both datasets
all_months = sorted(set(sleepstate_month_counts.index) | set(sleep_month_counts.index))

# Align data for plotting
sleepstate_values = [sleepstate_month_counts.get(month, 0) for month in all_months]
sleep_values = [sleep_month_counts.get(month, 0) for month in all_months]

# Create bar plot
x = np.arange(len(all_months))
width = 0.35

plt.bar(x - width/2, sleepstate_values, width, label='SleepStateBinary', alpha=0.7)
plt.bar(x + width/2, sleep_values, width, label='SleepBinary', alpha=0.7)

plt.xlabel('Month')
plt.ylabel('Number of Records')
plt.title('Distribution of Sleep Records by Month')
plt.xticks(x, [str(month) for month in all_months], rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary statistics
print("SleepStateBinary monthly distribution:")
print(sleepstate_month_counts.head(10))
print(f"\nTotal SleepStateBinary records: {len(df_SleepStateBin):,}")
print(f"Date range: {sleepstate_months.min()} to {sleepstate_months.max()}")

print("\nSleepBinary monthly distribution:")
print(sleep_month_counts.head(10))
print(f"\nTotal SleepBinary records: {len(df_SleepBin):,}")
print(f"Date range: {sleep_months.min()} to {sleep_months.max()}")

In [ ]:
# Filter data for 2025 only and get daily distribution
df_SleepStateBin_2025 = df_SleepStateBin[df_SleepStateBin["local_start_time"].dt.year == 2025]
df_SleepBin_2025 = df_SleepBin[df_SleepBin["local_start_time"].dt.year == 2025]

# Get the daily periods for both 2025 datasets
sleepstate_days_2025 = df_SleepStateBin_2025["local_start_time"].dt.date
sleep_days_2025 = df_SleepBin_2025["local_start_time"].dt.date

# Create histogram comparing the two for 2025 daily data
plt.figure(figsize=(20, 8))

# Count records per day
sleepstate_day_counts_2025 = pd.Series(sleepstate_days_2025).value_counts().sort_index()
sleep_day_counts_2025 = pd.Series(sleep_days_2025).value_counts().sort_index()

# Get all unique days across both datasets
all_days_2025 = sorted(set(sleepstate_day_counts_2025.index) | set(sleep_day_counts_2025.index))

# Align data for plotting
sleepstate_daily_values = [sleepstate_day_counts_2025.get(day, 0) for day in all_days_2025]
sleep_daily_values = [sleep_day_counts_2025.get(day, 0) for day in all_days_2025]

# Create bar plot
x_days = np.arange(len(all_days_2025))
width = 0.35

plt.bar(x_days - width/2, sleepstate_daily_values, width, label='SleepStateBinary', alpha=0.7)
plt.bar(x_days + width/2, sleep_daily_values, width, label='SleepBinary', alpha=0.7)

plt.xlabel('Day (2025)')
plt.ylabel('Number of Records')
plt.title('Distribution of Sleep Records by Day - 2025 Only')

# Format x-axis with fewer ticks to avoid overcrowding
tick_indices = range(0, len(all_days_2025), max(1, len(all_days_2025) // 20))  # Show ~20 ticks max
plt.xticks([x_days[i] for i in tick_indices], [str(all_days_2025[i]) for i in tick_indices], rotation=45)

plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary statistics for 2025 data
print("2025 SleepStateBinary daily distribution (first 10 days):")
print(sleepstate_day_counts_2025.head(10))
print(f"\nTotal SleepStateBinary records in 2025: {len(df_SleepStateBin_2025):,}")
if len(df_SleepStateBin_2025) > 0:
    print(f"Date range: {sleepstate_days_2025.min()} to {sleepstate_days_2025.max()}")

print("\n2025 SleepBinary daily distribution (first 10 days):")
print(sleep_day_counts_2025.head(10))
print(f"\nTotal SleepBinary records in 2025: {len(df_SleepBin_2025):,}")
if len(df_SleepBin_2025) > 0:
    print(f"Date range: {sleep_days_2025.min()} to {sleep_days_2025.max()}")

print(f"\nTotal unique days with data in 2025: {len(all_days_2025)}")

In [ ]:
# Merge the two DataFrames on customer and timestamp columns
sleep_comparison = df_SleepBin.merge(
    df_SleepStateBin, 
    on=['customer', 'local_start_time', 'local_end_time'], 
    how='outer',
    suffixes=('_SleepBin', '_SleepStateBin'),
    indicator=True
)

# Check the merge results
print("Merge indicator counts:")
print(sleep_comparison['_merge'].value_counts())

# Check if boolean values match where both exist
both_exist = sleep_comparison[sleep_comparison['_merge'] == 'both']
if len(both_exist) > 0:
    value_matches = both_exist['booleanValue_SleepBin'] == both_exist['booleanValue_SleepStateBin']
    
    if (~value_matches).sum() > 0:
        print("\nExamples of differing boolean values:")
        print(both_exist[~value_matches][['customer', 'local_start_time', 'local_end_time', 
                                         'booleanValue_SleepBin', 'booleanValue_SleepStateBin']].head())

# Summary
print(f"\nSummary:")
print(f"SleepBinary records: {len(df_SleepBin):_}")
print(f"SleepStateBinary records: {len(df_SleepStateBin):_}")
print(f"Records only in SleepBinary: {(sleep_comparison['_merge'] == 'left_only').sum():_}")
print(f"Records only in SleepStateBinary: {(sleep_comparison['_merge'] == 'right_only').sum():_}")
print(f"Records in both: {(sleep_comparison['_merge'] == 'both').sum():_}")

In [ ]:
left_only_records = sleep_comparison[sleep_comparison['_merge'] == 'left_only']
left_only_records

In [ ]:
row = left_only_records.iloc[1]
df_bup_sleep[
      (df_bup_sleep["customer"]       == row["customer"])
    & (df_bup_sleep["local_start_time"] >= row["local_start_time"])
    & (df_bup_sleep["local_start_time"] < row["local_end_time"])
    # & (df_bup_sleep["local_end_time"]   <= row["local_end_time"])
]


In [ ]:
# df_bup_sleep[(df_bup_sleep["local_start_time"].dt.floor("d") == pd.to_datetime("2025-03-04", utc=True)) & (df_bup_sleep["customer"] == "1Tii")]
df_bup_sleep[(df_bup_sleep["local_start_time"].dt.floor("d") == pd.to_datetime("2025-03-03", utc=True)) & (df_bup_sleep["customer"] == "SZPe")]

we can therefore drop SleepBinary, outside of some (only 53 records) edge cases, 
most of them are the same up to some point
and later developer changed to use SleepStateBinary only

## Sleep main

In [ ]:
df_bup_sleep = df_backup[df_backup["type"].isin([
    "SleepDeepBinary",
    "SleepLightBinary",
    # "SleepREMBinary", # only 4 people got these records
    # "SnoringBinary", # only 2 people got these records #? maybe relevant for this 2 people?
    "SleepStateBinary",
    # "SleepBinary", # is included in SleepStateBinary
    "SleepInBedBinary",
    "SleepAwakeBinary"
])]

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(df_bup_sleep[df_bup_sleep["type"] == "SleepStateBinary"]["local_start_time"].dt.hour, 
         bins=24, range=(0, 24), alpha=0.7, edgecolor='black', label='Start Time')
plt.hist(df_bup_sleep[df_bup_sleep["type"] == "SleepStateBinary"]["local_end_time"].dt.hour, 
         bins=24, range=(0, 24), alpha=0.7, edgecolor='black', label='End Time')
plt.title('Distribution of SleepStateBinary Start and End Hours (Local Time)')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Frequency')
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_bup_sleep.groupby('type', observed=True)["type"].nunique()

In [ ]:
(df_bup_sleep.groupby("type", observed=True)["start_end"]).describe() / 60

check if we can round to the whole minutes

In [ ]:
df_bup_sleep["local_start_time"].dt.second.describe([0.9, 0.95, 0.99, 0.999]) 

In [ ]:
df_bup_sleep["local_end_time"].dt.second.describe([0.9, 0.95, 0.99, 0.999]) 


In [ ]:
(df_bup_sleep["start_end"] % 60).describe([0.9, 0.95, 0.99, 0.999])

based on the above, round the `local_start_time`, `local_end_time` and `start_end` to the nearest minute

In [ ]:
df_bup_sleep["duration_preround"] = (df_bup_sleep["local_end_time"] - df_bup_sleep["local_start_time"]).dt.total_seconds()
df_bup_sleep["local_start_time"] = df_bup_sleep["local_start_time"].dt.round('min')
df_bup_sleep["local_end_time"] = df_bup_sleep["local_end_time"].dt.round('min')
df_bup_sleep["startTimestamp"] = df_bup_sleep["startTimestamp"].dt.round('min')
df_bup_sleep["endTimestamp"] = df_bup_sleep["endTimestamp"].dt.round('min')
df_bup_sleep["duration"] = (df_bup_sleep["local_end_time"] - df_bup_sleep["local_start_time"]).dt.total_seconds()

In [ ]:
(df_bup_sleep["duration"] - df_bup_sleep["duration_preround"]).describe([0.001, 0.01, 0.99, 0.999])

^ fine with this approximation: 98% is perfectly 0; 99.8% is +-1 second, and max is 30s

In [ ]:
df_bup_sleep.head()

In [ ]:
df_bup_sleep["duration"].describe([0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

In [ ]:
df_bup_sleep[df_bup_sleep.customer == "Vg3C"]

In [ ]:
# # Expand df_bup_sleep to have a record for each second using the established pattern
# df_bup_sleep_expanded = df_bup_sleep.loc[
#     df_bup_sleep.index.repeat(df_bup_sleep["duration"] / 60)
# ].copy()
# df_bup_sleep_expanded["time_offset"] = df_bup_sleep_expanded.groupby(level=0).cumcount()
# df_bup_sleep_expanded["timestamp"] = df_bup_sleep_expanded[
#     "local_start_time"
# ] + pd.to_timedelta(df_bup_sleep_expanded["time_offset"], unit="min")

# print(f"Original df_bup_sleep shape: {df_bup_sleep.shape}")
# print(f"Expanded df_bup_sleep shape: {df_bup_sleep_expanded.shape}")
# print(
#     f"Expansion factor: {df_bup_sleep_expanded.shape[0] / df_bup_sleep.shape[0]:.1f}x"
# )

# df_bup_sleep_expanded[
#     ["customer", "type", "timestamp", "local_start_time", "local_end_time"]
# ]

In [ ]:
# df_bup_sleep_expanded

In [ ]:
# df_bup_sleep_expanded = df_bup_sleep_expanded[:1_000_000]

In [ ]:
# # Convert to wide form where each sleep type becomes its own column
# # First, let's see what sleep types we have
# print("Sleep types in the data:")
# print(df_bup_sleep_expanded['type'].unique())

# # Create the wide form DataFrame
# df_sleep_wide = df_bup_sleep_expanded.pivot_table(
#     index=['customer', 'timestamp'], 
#     columns='type', 
#     values='booleanValue', 
#     aggfunc='max',  # Use max in case of overlapping intervals
#     fill_value=False
# ).reset_index()

# # Flatten column names
# df_sleep_wide.columns.name = None

# print(f"\nWide form DataFrame shape: {df_sleep_wide.shape}")
# print(f"Columns: {list(df_sleep_wide.columns)}")

# df_sleep_wide.head(10)

In [ ]:
# df_sleep_wide["binsum"] = df_sleep_wide["SleepAwakeBinary"].astype(int) + df_sleep_wide["SleepDeepBinary"].astype(int) + df_sleep_wide["SleepLightBinary"].astype(int)
# df_sleep_wide[df_sleep_wide["binsum"] > 1]

In [ ]:
df_bup_sleep.customer.unique()

In [ ]:
# start with one customer for testing
df = df_bup_sleep[df_bup_sleep.customer.isin(["Vg3C", "GE6w"])]
df

In [ ]:
# plot the data for this customer
import matplotlib.dates as mdates

plt.figure(figsize=(15, 6))
df_plot = df[df["customer"] == "Vg3C"]

# Define order and colors for sleep stages
stage_y_map = {
    "SleepDeepBinary": 0,
    "SleepLightBinary": 1,
    "SleepAwakeBinary": 2,
    "SleepInBedBinary": 3,
    "SleepStateBinary": 4,
}
colors = {
    "SleepDeepBinary": "navy",
    "SleepLightBinary": "royalblue",
    "SleepAwakeBinary": "orange",
    "SleepInBedBinary": "lightgray",
    "SleepStateBinary": "purple",
}

# Plot each stage using hlines (horizontal lines)
for stage, y_val in stage_y_map.items():
    stage_data = df_plot[df_plot["type"] == stage]
    if not stage_data.empty:
        for _, row in stage_data.iterrows():
            plt.hlines(
                y=y_val,
                xmin=row["local_start_time"],
                xmax=row["local_end_time"],
                linewidth=15,
                color=colors.get(stage, "black")
            )
        # Add label only once per stage
        plt.plot([], [], color=colors.get(stage, "black"), linewidth=15, label=stage)

start = pd.to_datetime("2025-02-12T21:00")
end = pd.to_datetime("2025-02-13T09:00")
plt.xlim(start, end)

plt.yticks(list(stage_y_map.values()), list(stage_y_map.keys()))
plt.xlabel("Local Time")
plt.title(f"Sleep Stages for Customer {df_plot['customer'].iloc[0] if not df_plot.empty else 'Unknown'}")
plt.grid(True, axis="x", alpha=0.3)
plt.legend()

# Format x-axis dates
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d %H:%M"))
plt.gcf().autofmt_xdate()

plt.tight_layout()
plt.show()

In [ ]:
start = pd.to_datetime("2025-02-01T21:00")
end = pd.to_datetime("2025-02-03T12:00")
df[(df["local_start_time"] >= start) & (df["local_end_time"] <= end) & (df["type"] == "SleepInBedBinary")]

## serious thing

In [ ]:
# df = df_bup_sleep
df = df_bup_sleep[df_bup_sleep.customer.isin(df_bup_sleep.customer.unique()[-10:])]
df = df.sort_values(by=["customer", "startTimestamp"]).reset_index(drop=True)

df_sleepinbed = df[df["type"] == "SleepInBedBinary"].sort_values(by=["customer", "startTimestamp"]).reset_index(drop=True)
df_sleepinbed["next_SleepInBed_start"] = df_sleepinbed.groupby("customer")["startTimestamp"].shift(-1)
df_sleepinbed["gap_to_next"] = df_sleepinbed["next_SleepInBed_start"] - df_sleepinbed["endTimestamp"]

MAX_GAP_SECOND = 90 * 60  # 60 minutes as the maximum gap to consider same sleep session
#? this can contain periods out of bed during sleep sessions

df_sleepinbed["is_lastentryinsession"] = (df_sleepinbed["gap_to_next"].dt.total_seconds() > MAX_GAP_SECOND) | (df_sleepinbed["gap_to_next"].isna())
df_sleepinbed["is_firstentryinsession"] = df_sleepinbed.groupby("customer")["is_lastentryinsession"].shift(1).fillna(True)
# TODO decide which session_id definition to use
# df_sleepinbed["sleep_session_id"] = df_sleepinbed.groupby("customer")["is_firstentryinsession"].cumsum()
df_sleepinbed["sleep_session_id"] = df_sleepinbed["is_firstentryinsession"].cumsum() - 1 #-1 to start from 0

print(df_sleepinbed.customer.nunique())
df_sleepinbed


In [ ]:
df_sleep_sessions = (
    df_sleepinbed.groupby(["customer", "sleep_session_id"])
    .agg(
        {
            "startTimestamp": "first", # it's sorted already
            "endTimestamp": "last",
            "local_start_time": "first",
            "local_end_time": "last",
        }
    )
    .reset_index()
)
df_sleep_sessions["sleep_session_duration"] = (df_sleep_sessions["endTimestamp"] - df_sleep_sessions["startTimestamp"]).dt.total_seconds()
df_sleep_sessions

| Done | ID | Variable Name | Description | Notes |
| :---: | :---: | :--- | :--- | :--- |
| [x] | 11 | `total_sleep_time` | Total sleep time, i.e., sum of all “non-awake” stages, in seconds | |
| [x] | 12 | `time_in_bed` | Total time in bed, i.e., sum of all detected stages (including awake stages), in seconds; interval between sleep_onset and sleep_offset | |
| [x] | 13 | `light_pct` | Proportion of total time in bed spent in light sleep stage | |
| [x] | 14 | `deep_pct` | Proportion of total time in bed spent in deep sleep stage | |
| ~~[ ]~~ | 15 | ~~`REM_pct`~~| ~~Proportion of total time in bed spent in REM (rapid eye movement) sleep stage~~| we don't have REM data, there's only for 10 patients, the developer algorithm is probably wrong :(|
| [x] | 16 | `awake_pct` | Proportion of total time in bed spent awake | |
| [ ] | 17 | ~~`NREM_pct`~~ | ~~Proportion of total time in bed spent in a non-REM (non-rapid eye movement) sleep stage~~ | |
| [x] | 18 | `sleep_onset` | Exact onset time (in hours local time as decimal number, e.g., 22.5 = 10:30 p.m.) of sleep record, i.e., start of first “non-awake” stage | |
| [x] | 19 | `sleep_offset` | Exact offset time (in hours local time as decimal number, e.g., 7.25 = 7:15 a.m.) of sleep record, i.e., end of last “non-awake” stage | |
| ~~[ ]~~ | 20 | ~~`rem_latency`~~ | ~~Interval in hours between sleep onset and the occurrence of the first REM stage~~ | |
| [x] | 21 | `sleep_efficiency` | Percentage of total sleep time to time in bed | |
| [x] | 22 | `awakenings` | Number of episodes in which the participant is awake for more than 5 minutes | |
| [x] | 23 | `insomnia` | 1 if total sleep time < 6 hours and at least one awakening of more than 30 minutes | |
| [x] | 24 | `hypersomnia` | 1 if total sleep time > 10 hours | |

In [ ]:
from tqdm import tqdm

AWAKENING_THRESHOLD = 5 * 60  # 5 minutes]
INSOMNIA_SLEEP_THRESHOLD = 6 * 60 * 60  # 6 hours
INSOMNIA_AWAKE_THRESHOLD = 30 * 60  # 30 minutes
HYPERSOMNIA_THRESHOLD = 10 * 60 * 60  # 10 hours

# Initialize columns if they don't exist
for col in [
    "total_sleep_time",
    "time_in_bed",
    "time_out_of_bed",
    "SleepAwake_duration",
    "SleepLight_duration",
    "SleepDeep_duration",
    "sleep_onset",
    "sleep_offset",
    "sleep_efficiency",
    "awakenings",
    # "long_awakings",
    "insomnia",
    "hypersomnia",
]:
    if col not in df_sleep_sessions.columns:
        df_sleep_sessions[col] = pd.NA

for sleep_ses_id in tqdm(range(len(df_sleep_sessions))):
# for sleep_ses_id in [56]:
    session = df_sleep_sessions.iloc[sleep_ses_id]
    ses = df[
        (df.customer == session.customer)
        & (df.startTimestamp >= session.startTimestamp)
        & (df.endTimestamp <= session.endTimestamp)
    ]

    # awakenings
    ses_sleepstate = ses[ses.type == "SleepStateBinary"]
    ses_sleepstate["next_SleepState_start"] = ses_sleepstate["startTimestamp"].shift(-1)
    ses_sleepstate["gap_to_next"] = ses_sleepstate["next_SleepState_start"] - ses_sleepstate["endTimestamp"]
    ses_sleepstate["is_awakening"] = ses_sleepstate["gap_to_next"].dt.total_seconds() >= AWAKENING_THRESHOLD
    ses_sleepstate["is_long_awakening"] = ses_sleepstate["gap_to_next"].dt.total_seconds() >= INSOMNIA_AWAKE_THRESHOLD
    num_awakenings = ses_sleepstate["is_awakening"].sum()
    num_long_awakenings = ses_sleepstate["is_long_awakening"].sum()

    # Use .loc for proper assignment
    df_sleep_sessions.loc[df_sleep_sessions.index[sleep_ses_id], "total_sleep_time"] = (
        ses[ses.type == "SleepStateBinary"].duration.sum()
    )
    df_sleep_sessions.loc[df_sleep_sessions.index[sleep_ses_id], "time_in_bed"] = ses[
        ses.type == "SleepInBedBinary"
    ].duration.sum()
    df_sleep_sessions.loc[
        df_sleep_sessions.index[sleep_ses_id], "SleepAwake_duration"
    ] = ses[ses.type == "SleepAwakeBinary"].duration.sum()
    df_sleep_sessions.loc[
        df_sleep_sessions.index[sleep_ses_id], "SleepLight_duration"
    ] = ses[ses.type == "SleepLightBinary"].duration.sum()
    df_sleep_sessions.loc[
        df_sleep_sessions.index[sleep_ses_id], "SleepDeep_duration"
    ] = ses[ses.type == "SleepDeepBinary"].duration.sum()

    df_sleep_sessions.loc[
        df_sleep_sessions.index[sleep_ses_id], "sleep_onset"
    ] = ses_sleepstate["local_start_time"].min()
    df_sleep_sessions.loc[
        df_sleep_sessions.index[sleep_ses_id], "sleep_offset"
    ] = ses_sleepstate["local_end_time"].max()

    df_sleep_sessions.loc[
        df_sleep_sessions.index[sleep_ses_id], "awakenings"
    ] = num_awakenings
    # df_sleep_sessions.loc[
    #     df_sleep_sessions.index[sleep_ses_id], "long_awakings"
    # ] = num_long_awakenings
    df_sleep_sessions.loc[
        df_sleep_sessions.index[sleep_ses_id], "insomnia"
    ] = (
        (df_sleep_sessions.loc[df_sleep_sessions.index[sleep_ses_id], "total_sleep_time"] <= INSOMNIA_SLEEP_THRESHOLD)
        & (num_long_awakenings >= 1)
    )

# TODO should i change time_in_bed to sleep_state duration?
#? should it include out-of-bed periods?
df_sleep_sessions["time_out_of_bed"] = (
    df_sleep_sessions["sleep_session_duration"] - df_sleep_sessions["time_in_bed"]
)
df_sleep_sessions["sleep_efficiency"] = (
    df_sleep_sessions["total_sleep_time"] / df_sleep_sessions["time_in_bed"]
)
df_sleep_sessions["hypersomnia"] = df_sleep_sessions["total_sleep_time"] >= HYPERSOMNIA_THRESHOLD

df_sleep_sessions["awake_pct"] = df_sleep_sessions["SleepAwake_duration"] / df_sleep_sessions["time_in_bed"]
df_sleep_sessions["light_sleep_pct"] = df_sleep_sessions["SleepLight_duration"] / df_sleep_sessions["time_in_bed"]
df_sleep_sessions["deep_sleep_pct"] = df_sleep_sessions["SleepDeep_duration"] / df_sleep_sessions["time_in_bed"]

df_sleep_sessions_original = df_sleep_sessions.copy()
df_sleep_sessions_original


In [ ]:
df = df_bup_sleep.sort_values(by=["customer", "startTimestamp"]).reset_index(drop=True)
# df = (
#     df_bup_sleep[df_bup_sleep.customer.isin(df_bup_sleep.customer.unique()[-10:])]
#     .sort_values(by=["customer", "startTimestamp"])
#     .reset_index(drop=True)
# )


In [ ]:
# Complete sleep session aggregation - starts from df, produces df_sleep_sessions
# FULLY VECTORIZED - no loops!

AWAKENING_THRESHOLD = 5 * 60  # 5 minutes
INSOMNIA_SLEEP_THRESHOLD = 6 * 60 * 60  # 6 hours
INSOMNIA_AWAKE_THRESHOLD = 30 * 60  # 30 minutes
HYPERSOMNIA_THRESHOLD = 10 * 60 * 60  # 10 hours
MAX_GAP_SECOND = 90 * 60  # 90 minutes as the maximum gap to consider same sleep session
# Step 1: Identify sleep sessions from all the records
df_sleepall = (
    # df[df["type"] == "SleepInBedBinary"]
    df
    .sort_values(by=["customer", "startTimestamp"])
    .reset_index(drop=True)
)
df_sleepall["next_SleepInBed_start"] = df_sleepall.groupby("customer")[
    "startTimestamp"
].shift(-1)
df_sleepall["gap_to_next"] = (
    df_sleepall["next_SleepInBed_start"] - df_sleepall["endTimestamp"]
)
df_sleepall["is_lastentryinsession"] = (
    df_sleepall["gap_to_next"].dt.total_seconds() > MAX_GAP_SECOND
) | (df_sleepall["gap_to_next"].isna())
df_sleepall["is_firstentryinsession"] = (
    df_sleepall.groupby("customer")["is_lastentryinsession"].shift(1).fillna(True)
)
df_sleepall["sleep_session_id"] = df_sleepall["is_firstentryinsession"].cumsum() - 1

# Step 2: Create df_sleep_sessions with session boundaries
df_sleep_sessions = (
    df_sleepall.groupby(["customer", "sleep_session_id"])
    .agg(
        {
            "startTimestamp": "first",
            "endTimestamp": "last",
            "local_start_time": "first",
            "local_end_time": "last",
        }
    )
    .reset_index()
)
df_sleep_sessions["sleep_session_duration"] = (
    df_sleep_sessions["endTimestamp"] - df_sleep_sessions["startTimestamp"]
).dt.total_seconds()

# Step 3: Assign session IDs to all relevant records via merge + filter
relevant_types = [
    "SleepInBedBinary",
    "SleepAwakeBinary",
    "SleepLightBinary",
    "SleepDeepBinary",
    "SleepStateBinary",
]
df_relevant = df[df["type"].isin(relevant_types)].copy()

# Merge on customer only, then filter by time bounds (more memory but no sorting issues)
sessions_for_merge = df_sleep_sessions[
    ["customer", "sleep_session_id", "startTimestamp", "endTimestamp"]
].rename(columns={"startTimestamp": "session_start", "endTimestamp": "session_end"})

# df_with_session = df_relevant.merge(sessions_for_merge, on="customer", how="inner")
# df_with_session

df_with_session = (
    pd.merge_asof(
        df.sort_values(by=["startTimestamp", "customer"]),
        sessions_for_merge.sort_values(by=["session_start", "customer"]),
        left_on="startTimestamp",
        right_on="session_start",
        by="customer",
        direction="backward",
    )
    .sort_values(by=["customer", "startTimestamp"])
    .reset_index(drop=True)
)

# # Filter to records within their session bounds
# df_with_session = df_with_session[
#     (df_with_session["startTimestamp"] >= df_with_session["session_start"]) &
#     (df_with_session["endTimestamp"] <= df_with_session["session_end"])
# ].copy()

# Step 4: Aggregate durations by session and type (vectorized!)
duration_agg = (
    df_with_session.groupby(["customer", "sleep_session_id", "type"], observed=True)[
        "duration"
    ]
    .sum()
    .unstack(fill_value=0)
)

# Rename columns
col_mapping = {
    "SleepInBedBinary": "time_in_bed",
    "SleepAwakeBinary": "SleepAwake_duration",
    "SleepLightBinary": "SleepLight_duration",
    "SleepDeepBinary": "SleepDeep_duration",
    "SleepStateBinary": "total_sleep_time",
}
duration_agg = duration_agg.rename(columns=col_mapping)
# Ensure all columns exist
for col in col_mapping.values():
    if col not in duration_agg.columns:
        duration_agg[col] = 0
duration_agg = duration_agg.reset_index()

# Step 5: Calculate awakenings (vectorized with groupby + shift)
sleepstate = (
    df_with_session[df_with_session["type"] == "SleepStateBinary"]
    .sort_values(["customer", "sleep_session_id", "startTimestamp"])
    .copy()
)
sleepstate["next_start"] = sleepstate.groupby(["customer", "sleep_session_id"])[
    "startTimestamp"
].shift(-1)
sleepstate["gap_to_next"] = (
    sleepstate["next_start"] - sleepstate["endTimestamp"]
).dt.total_seconds()
sleepstate["is_awakening"] = sleepstate["gap_to_next"] >= AWAKENING_THRESHOLD
sleepstate["is_long_awakening"] = sleepstate["gap_to_next"] >= INSOMNIA_AWAKE_THRESHOLD

awakening_agg = (
    sleepstate.groupby(["customer", "sleep_session_id"])
    .agg(
        awakenings=("is_awakening", "sum"),
        long_awakenings=("is_long_awakening", "sum"),
        sleep_onset=("local_start_time", "min"),
        sleep_offset=("local_end_time", "max"),
    )
    .reset_index()
)

# Step 6: Merge all aggregations back to df_sleep_sessions
df_sleep_sessions = df_sleep_sessions.merge(
    duration_agg, on=["customer", "sleep_session_id"], how="left"
)
df_sleep_sessions = df_sleep_sessions.merge(
    awakening_agg, on=["customer", "sleep_session_id"], how="left"
)

# Fill NaN with 0 for numeric columns
for col in [
    "time_in_bed",
    "SleepAwake_duration",
    "SleepLight_duration",
    "SleepDeep_duration",
    "total_sleep_time",
    "awakenings",
    "long_awakenings",
]:
    df_sleep_sessions[col] = df_sleep_sessions[col].fillna(0)

#!!!! time_in_bed is wrongly computed for samples from 2023 (==0) as it wasn't recorded
# Step 7: Compute derived metrics
df_sleep_sessions["time_out_of_bed"] = (
    df_sleep_sessions["sleep_session_duration"] - df_sleep_sessions["time_in_bed"]
) #!
df_sleep_sessions["sleep_efficiency"] = df_sleep_sessions[
    "total_sleep_time"
] / df_sleep_sessions["time_in_bed"].replace(0, np.nan)
df_sleep_sessions["hypersomnia"] = (
    df_sleep_sessions["total_sleep_time"] >= HYPERSOMNIA_THRESHOLD
)
df_sleep_sessions["insomnia"] = (
    df_sleep_sessions["total_sleep_time"] <= INSOMNIA_SLEEP_THRESHOLD
) & (df_sleep_sessions["long_awakenings"] >= 1)

# !!
df_sleep_sessions["awake_pct"] = df_sleep_sessions[
    "SleepAwake_duration"
] / df_sleep_sessions["time_in_bed"].replace(0, np.nan)
df_sleep_sessions["light_sleep_pct"] = df_sleep_sessions[
    "SleepLight_duration"
] / df_sleep_sessions["time_in_bed"].replace(0, np.nan)
df_sleep_sessions["deep_sleep_pct"] = df_sleep_sessions[
    "SleepDeep_duration"
] / df_sleep_sessions["time_in_bed"].replace(0, np.nan)

# Reorder columns to match expected order
column_order = [
    "customer",
    "sleep_session_id",
    "startTimestamp",
    "endTimestamp",
    "local_start_time",
    "local_end_time",
    "sleep_session_duration",
    "total_sleep_time",
    "time_in_bed",
    "time_out_of_bed",
    "SleepAwake_duration",
    "SleepLight_duration",
    "SleepDeep_duration",
    "sleep_onset",
    "sleep_offset",
    "sleep_efficiency",
    "awakenings",
    "long_awakenings",
    "insomnia",
    "hypersomnia",
    "awake_pct",
    "light_sleep_pct",
    "deep_sleep_pct",
]
df_sleep_sessions = df_sleep_sessions[
    [col for col in column_order if col in df_sleep_sessions.columns]
]

df_sleep_sessions

In [76]:
df_sleep_sessions

,customer,sleep_session_id,startTimestamp,endTimestamp,local_start_time,local_end_time,sleep_session_duration,total_sleep_time,time_in_bed,time_out_of_bed,...,sleep_onset,sleep_offset,sleep_efficiency,awakenings,long_awakenings,insomnia,hypersomnia,awake_pct,light_sleep_pct,deep_sleep_pct
0,05kz,0,2023-10-10 22:13:00+00:00,2023-10-11 06:00:00+00:00,2023-10-11 00:13:00,2023-10-11 08:00:00,28020.0,27240.0,27240.0,780.0,...,2023-10-11 00:15:00,2023-10-11 07:59:00,1.000000,1.0,0.0,False,False,0.028634,0.447137,0.552863
1,05kz,1,2023-10-11 21:11:00+00:00,2023-10-12 08:00:00+00:00,2023-10-11 23:11:00,2023-10-12 10:00:00,38940.0,33780.0,33780.0,5160.0,...,2023-10-11 23:13:00,2023-10-12 10:00:00,1.000000,3.0,1.0,False,False,0.152753,0.451155,0.550622
2,05kz,2,2023-10-12 21:10:00+00:00,2023-10-13 06:14:00+00:00,2023-10-12 23:10:00,2023-10-13 08:14:00,32640.0,30660.0,30660.0,1980.0,...,2023-10-12 23:12:00,2023-10-13 08:10:00,1.000000,2.0,0.0,False,False,0.064579,0.338552,0.661448
3,05kz,3,2023-10-14 23:48:00+00:00,2023-10-15 04:30:00+00:00,2023-10-15 01:48:00,2023-10-15 06:30:00,16920.0,16800.0,16800.0,120.0,...,2023-10-15 01:50:00,2023-10-15 06:30:00,1.000000,0.0,0.0,False,False,0.007143,0.489286,0.510714
4,05kz,4,2023-10-15 21:11:00+00:00,2023-10-16 07:28:00+00:00,2023-10-15 23:11:00,2023-10-16 09:28:00,37020.0,35640.0,35640.0,1380.0,...,2023-10-15 23:13:00,2023-10-16 09:28:00,1.000000,3.0,0.0,False,False,0.038721,0.498316,0.501684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94069,zgxc,94069,2024-06-21 23:20:00+00:00,2024-06-22 08:07:00+00:00,2024-06-22 01:20:00,2024-06-22 10:07:00,31620.0,30120.0,31620.0,0.0,...,2024-06-22 01:22:00,2024-06-22 10:07:00,0.952562,2.0,0.0,False,False,0.047438,0.679317,0.273245
94070,zgxc,94070,2024-06-23 01:11:00+00:00,2024-06-23 09:44:00+00:00,2024-06-23 03:11:00,2024-06-23 11:44:00,30780.0,29820.0,30780.0,0.0,...,2024-06-23 03:13:00,2024-06-23 11:39:00,0.968811,1.0,0.0,False,False,0.031189,0.779727,0.189084
94071,zgxc,94071,2024-06-23 22:08:00+00:00,2024-06-24 06:11:00+00:00,2024-06-24 00:08:00,2024-06-24 08:11:00,28980.0,27660.0,28980.0,0.0,...,2024-06-24 00:11:00,2024-06-24 08:09:00,0.954451,2.0,0.0,False,False,0.045549,0.712215,0.242236
94072,zgxc,94072,2024-06-24 23:37:00+00:00,2024-06-25 05:27:00+00:00,2024-06-25 01:37:00,2024-06-25 07:27:00,21000.0,20460.0,21000.0,0.0,...,2024-06-25 01:39:00,2024-06-25 07:27:00,0.974286,1.0,0.0,False,False,0.025714,0.748571,0.225714


In [ ]:
import pandas as pd

# Test if df_sleep_sessions equals df_sleep_sessions_original

# Check if both dataframes have the same shape
print(f"Shape comparison:")
print(f"df_sleep_sessions: {df_sleep_sessions.shape}")
print(f"df_sleep_sessions_original: {df_sleep_sessions_original.shape}")

# Sort both dataframes to ensure same order
df_sorted = df_sleep_sessions.sort_values(['customer', 'sleep_session_id']).reset_index(drop=True)
df_original_sorted = df_sleep_sessions_original.sort_values(['customer', 'sleep_session_id']).reset_index(drop=True)

# Try using pandas testing utility
try:
    pd.testing.assert_frame_equal(df_sorted, df_original_sorted, check_dtype=False)
    print("\n✓ DataFrames are equal!")
except AssertionError as e:
    print(f"\n✗ DataFrames are NOT equal:")
    print(e)
    
    # Show differences in columns if they exist
    if set(df_sorted.columns) != set(df_original_sorted.columns):
        print("\nColumn differences:")
        print(f"Only in df_sleep_sessions: {set(df_sorted.columns) - set(df_original_sorted.columns)}")
        print(f"Only in df_sleep_sessions_original: {set(df_original_sorted.columns) - set(df_sorted.columns)}")

In [ ]:
pd.testing.assert_frame_equal(df_sorted, df_original_sorted, check_dtype=False)


In [90]:
# check if the time adds up
df_tmp = df_sleep_sessions[df_sleep_sessions["time_in_bed"] > 0]
df_tmp2 = df_tmp[(df_tmp["total_sleep_time"] + df_tmp["SleepAwake_duration"] != df_tmp["time_in_bed"])]
((df_tmp2["total_sleep_time"] + df_tmp2["SleepAwake_duration"] - df_tmp2["time_in_bed"])/3600).describe()

count    9007.000000
mean        0.432647
std         0.630101
min         0.016667
25%         0.150000
50%         0.300000
75%         0.550000
max        11.766667
dtype: float64

In [92]:
df_tmp = df_sleep_sessions
(df_tmp["SleepDeep_duration"] + df_tmp["SleepLight_duration"] - df_tmp["total_sleep_time"]).describe()

count    94074.000000
mean        -0.510237
std       1216.971708
min     -36780.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      36300.000000
dtype: float64

In [84]:
df_sleep_sessions[df_sleep_sessions["time_in_bed"] == 0]

,customer,sleep_session_id,startTimestamp,endTimestamp,local_start_time,local_end_time,sleep_session_duration,total_sleep_time,time_in_bed,time_out_of_bed,...,sleep_onset,sleep_offset,sleep_efficiency,awakenings,long_awakenings,insomnia,hypersomnia,awake_pct,light_sleep_pct,deep_sleep_pct
497,0NEG,497,2023-10-09 22:51:00+00:00,2023-10-10 08:35:00+00:00,2023-10-10 00:51:00,2023-10-10 10:35:00,35040.0,32820.0,0.0,35040.0,...,2023-10-10 00:53:00,2023-10-10 10:04:00,NaN,0.0,0.0,False,False,NaN,NaN,NaN
524,0ePW,524,2023-07-03 22:42:00+00:00,2023-07-04 08:47:00+00:00,2023-07-04 00:42:00,2023-07-04 10:47:00,36300.0,34140.0,0.0,36300.0,...,2023-07-04 00:44:00,2023-07-04 10:47:00,NaN,2.0,0.0,False,False,NaN,NaN,NaN
525,0ePW,525,2023-07-04 22:35:00+00:00,2023-07-05 10:28:00+00:00,2023-07-05 00:35:00,2023-07-05 12:28:00,42780.0,39660.0,0.0,42780.0,...,2023-07-05 00:37:00,2023-07-05 12:28:00,NaN,3.0,0.0,False,True,NaN,NaN,NaN
526,0ePW,526,2023-07-06 00:28:00+00:00,2023-07-06 07:02:00+00:00,2023-07-06 02:28:00,2023-07-06 09:02:00,23640.0,22860.0,0.0,23640.0,...,2023-07-06 02:30:00,2023-07-06 09:02:00,NaN,1.0,0.0,False,False,NaN,NaN,NaN
527,0ePW,527,2023-07-06 22:53:00+00:00,2023-07-07 09:51:00+00:00,2023-07-07 00:53:00,2023-07-07 11:51:00,39480.0,35340.0,0.0,39480.0,...,2023-07-07 00:55:00,2023-07-07 11:51:00,NaN,1.0,1.0,False,False,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89629,xlB8,89629,2023-09-22 21:24:00+00:00,2023-09-23 06:58:00+00:00,2023-09-22 22:24:00,2023-09-23 07:58:00,34440.0,30360.0,0.0,34440.0,...,2023-09-22 22:26:00,2023-09-23 07:56:00,NaN,3.0,0.0,False,False,NaN,NaN,NaN
89630,xlB8,89630,2023-09-23 23:05:00+00:00,2023-09-24 07:16:00+00:00,2023-09-24 00:05:00,2023-09-24 08:16:00,29460.0,27060.0,0.0,29460.0,...,2023-09-24 00:07:00,2023-09-24 08:14:00,NaN,3.0,0.0,False,False,NaN,NaN,NaN
89637,xlB8,89637,2023-09-30 22:59:00+00:00,2023-10-01 08:17:00+00:00,2023-09-30 23:59:00,2023-10-01 09:17:00,33480.0,29160.0,0.0,33480.0,...,2023-10-01 00:01:00,2023-10-01 09:17:00,NaN,3.0,0.0,False,False,NaN,NaN,NaN
91346,xwB7,91346,2023-07-20 20:39:00+00:00,2023-07-21 06:53:00+00:00,2023-07-20 22:39:00,2023-07-21 08:53:00,36840.0,34860.0,0.0,36840.0,...,2023-07-20 22:41:00,2023-07-21 08:53:00,NaN,2.0,0.0,False,False,NaN,NaN,NaN


In [94]:
df.groupby("type", observed=True)["startTimestamp"].min()

type
SleepAwakeBinary   2023-05-17 20:38:00+00:00
SleepDeepBinary    2023-05-17 21:14:00+00:00
SleepInBedBinary   2023-09-07 19:47:00+00:00
SleepLightBinary   2023-05-17 20:40:00+00:00
SleepStateBinary   2023-05-17 20:40:00+00:00
Name: startTimestamp, dtype: datetime64[ns, UTC]